In [ ]:
from config import (BOOTSTRAP_RESAMPLES, RANDOM_SEED, FOCAL_AUTHOR_ID,
                    BUNDLED_LAUREATES, input_file, output_file, margin_file)

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
aid = FOCAL_AUTHOR_ID

In [ ]:
author2paper = pd.read_csv(input_file('Paper_Author.csv.zip') , usecols=['PaperID', 'AuthorID'])
author2paper = author2paper.drop_duplicates().reset_index(drop=True)

In [ ]:
pubs = author2paper[author2paper['AuthorID'] == aid]
coauthordf = author2paper[author2paper['PaperID'].isin(pubs['PaperID'].unique()) & (author2paper['AuthorID'] != aid)].copy().reset_index(drop=True)

In [ ]:
paper2year = pd.read_csv(input_file('Paper_year.csv.zip'))

In [ ]:
paper_year = paper2year.set_index('PaperID')['Year']
career_start_year = paper_year.reindex(pubs['PaperID']).min()
if pd.isna(career_start_year):
    raise ValueError('No publication year is available for the focal author.')
coauthordf['Year'] = coauthordf['PaperID'].map(paper_year)
coauthordf['Focal_AID'] = aid
coauthordf['t'] = coauthordf['Year'] - career_start_year

# Load data

In [ ]:
def set_fig_(ax , xlabel=0, ylabel=0 , xlog=0, ylog=0, fontsize=16, iflegend=0, legendloc=2):
    if xlabel!=0:
        ax.set_xlabel(xlabel,size = fontsize)
    else:
        ax.set_xlabel('')
    if ylabel!=0:
        ax.set_ylabel(ylabel,size = fontsize)
    else:
        ax.set_ylabel('')
    if ylog:
        ax.set_yscale("log")
    if xlog:
        ax.set_xscale("log")
    if iflegend==1:
        ax.legend(frameon =False,fontsize = fontsize, loc=legendloc)
    if iflegend==2:
        ax.legend(frameon =False,fontsize = fontsize,ncol=1,bbox_to_anchor=(1, 0), loc=3, borderaxespad=0)
    ax.tick_params(axis='x',labelsize=fontsize)
    ax.tick_params(axis='y',labelsize=fontsize)
    plt.tick_params(axis='both', which='major', labelsize=16)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.spines["right"].set_linewidth(2)
    ax.spines["top"].set_linewidth(2)

In [ ]:
df = pd.read_csv(input_file('Paper_nref.csv.zip'))
coauthordf['Reference_Count'] = coauthordf['PaperID'].map(df.set_index('PaperID')['Reference_Count'])

df = pd.read_csv(input_file('Paper_nau.csv.zip'))
coauthordf['Team_Size'] = coauthordf['PaperID'].map(df.set_index('PaperID')['Team_Size'])

In [ ]:
df = pd.read_csv(input_file('Paper_ncit.csv.zip'))
coauthordf['cit_pct'] = coauthordf['PaperID'].map(df.set_index('PaperID')['cit_pct'])
coauthordf['cit_pct'] = coauthordf['cit_pct'].fillna(0)

df = pd.read_csv(input_file('Paper_Disruption.csv.zip'))
coauthordf['CD_pct'] = coauthordf['PaperID'].map(df.set_index('PaperID')['CD_pct'])

df = pd.read_csv(input_file('Paper_Atypicality.csv.zip'))
coauthordf['Novelty_pct'] = coauthordf['PaperID'].map(df.set_index('PaperID')['Novelty_pct'])

# Illustration figure

In [ ]:
data = coauthordf.groupby('AuthorID')['t'].min().reset_index()
data.columns = ['AuthorID','t_start']

maxyear = coauthordf.groupby('AuthorID')['t'].max().reset_index()
data['t_end'] = maxyear['t']

data['l'] = data['t_end'] - data['t_start']+1

n_k = coauthordf.groupby('AuthorID')['PaperID'].nunique().reset_index()
data['k'] = data['AuthorID'].map(n_k.set_index('AuthorID')['PaperID'])

In [ ]:
avg_teamsize = coauthordf.groupby('AuthorID')['Team_Size'].mean().reset_index()
data['Team_Size'] = avg_teamsize['Team_Size']

avg_Novelty = coauthordf.groupby('AuthorID')['Novelty_pct'].mean().reset_index()
data['Novelty_pct'] = avg_Novelty['Novelty_pct']

avg_cit = coauthordf.groupby('AuthorID')['cit_pct'].mean().reset_index()
data['cit_pct'] = avg_cit['cit_pct']

avg_CD = coauthordf.groupby('AuthorID')['CD_pct'].mean().reset_index()
data['CD_pct'] = avg_CD['CD_pct']

In [ ]:
data['l2'] = data['l'].clip(upper=10)
data = data.sort_values('t_start').reset_index(drop=True)
laureates = pd.read_csv(BUNDLED_LAUREATES)
prize_rows = laureates.loc[laureates['AuthorID'] == aid, 'Prize year']
if len(prize_rows) != 1:
    raise ValueError('The focal author needs exactly one prize year in the bundled table.')
Prizeyear = int(prize_rows.iloc[0])

In [ ]:
colors = sns.color_palette("viridis",9)
fig = plt.figure(figsize=(16 , 12))
gs = gridspec.GridSpec(3, 4, figure=fig, width_ratios=[1 ,1 , 1 , 1], height_ratios=[3, 3, 2], wspace=0.35, hspace=0.35)

ax = fig.add_subplot(gs[:2, :2])
plt.axvline(x = Prizeyear - career_start_year, color='tab:red', linestyle='--')
plt.scatter(data[data['l2']==1]['t_start'].values , data[data['l2']==1].index , color = 'grey' , s=data[data['l2']==1]['k'].values*5)
for i in range(9):
    plt.scatter(data[data['l2']==i+2]['t_start'].values , data[data['l2']==i+2].index , color = colors[i] , s=data[data['l2']==i+2]['k'].values*5)
    plt.scatter(data[data['l2']==i+2]['t_end'].values , data[data['l2']==i+2].index , color = colors[i] , s=data[data['l2']==i+2]['k'].values*5)
    for j in range(len(data[data['l2']==i+2].index)):
        if data[data['l2']==i+2]['t_end'].values[j] != data[data['l2']==i+2]['t_start'].values[j]:
            plt.hlines(data[data['l2']==i+2].index[j] , data[data['l2']==i+2]['t_start'].values[j], data[data['l2']==i+2]['t_end'].values[j] , color = colors[i])
ax_top = ax.secondary_xaxis('top')
ax_top.set_xticks([0,10,20,30,40,50] , np.array(np.array([0,10,20,30,40,50]) + career_start_year).astype(int))
ax_top.tick_params(axis='x', labelsize=16)
set_fig_(ax , xlabel='Career age (t)', ylabel='Collaborator order', xlog=0, iflegend=0)

ax = fig.add_subplot(gs[0, 2])
data_ = data.sort_values('l',ascending=False).reset_index(drop=True)
plt.scatter(data_[data_['l2']==1].index , data_[data_['l2']==1]['l'].values , color = 'grey' , s = data_[data_['l2']==1]['k'].values*5)
for i in range(9):
    plt.scatter(data_[data_['l2']==i+2].index , data_[data_['l2']==i+2]['l'].values , color = colors[i] , s =  data_[data_['l2']==i+2]['k'].values*5)
set_fig_(ax , xlabel='Collaborator rank (by $l$)', ylabel='$l$', xlog=1, ylog=1)

ax = fig.add_subplot(gs[1, 2:])
cu1_n_t = coauthordf.groupby('t')['PaperID'].nunique().reset_index()
cu1_n_t['n_cum'] = cu1_n_t['PaperID'].cumsum()
plt.plot(cu1_n_t['t'].values , cu1_n_t['n_cum'].values , 'k-' , marker = 'D')
for coauid in coauthordf['AuthorID'].unique():
    coauthordf_ = coauthordf[coauthordf['AuthorID']==coauid]
    if len (coauthordf_) > 1:
        cu1_n_t_ = coauthordf_.groupby('t')['PaperID'].count().reset_index()
        cu1_n_t_['n_cum'] = cu1_n_t_['PaperID'].cumsum()
        l = cu1_n_t_['n_cum'].max()
        plt.plot(cu1_n_t_['t'].values , cu1_n_t_['n_cum'].values , color=colors[l] if l<=8 else colors[-1])
        plt.scatter(cu1_n_t_['t'].values , cu1_n_t_['n_cum'].values , color=colors[l] if l<=8 else colors[-1] , s = len(cu1_n_t_)*5)
    else:
        plt.plot(coauthordf_['t'].values , [1] , color='grey')
        plt.scatter(coauthordf_['t'].values , [1] , color='grey' , s = 10)
set_fig_(ax , xlabel='Career age (t)', ylabel='$k (t)$', xlog=0, ylog=1)

ax = fig.add_subplot(gs[0, 3])
ax.scatter(data['l'].values , data['k'].values , color = 'grey' , s = data['k'].values*5)
sns.regplot(data , x='l' , y = 'k' , scatter=False, n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.xlim(.8,51)
plt.ylim(.8,51)
set_fig_(ax , xlabel='$l$', ylabel='$k$', xlog=1, ylog=1, iflegend=0)
plt.savefig(output_file("Illustration_main.svg"),bbox_inches = 'tight')

## Legend

In [ ]:
fig = plt.figure(figsize=(20 , 12))
gs = gridspec.GridSpec(3, 4, figure=fig, wspace=.35, hspace=.35)
ax = fig.add_subplot(gs[0])
for i in [1,2,5,10,20,100]:
    plt.scatter(1,1,color = 'grey',s=i*5,label = '$k$ = '+str(i))
plt.xlim(2,4)
plt.ylim(2,4)
set_fig_(ax , iflegend=1)
plt.savefig(output_file("Illustration_legend1.svg"),bbox_inches = 'tight')

In [ ]:
fig = plt.figure(figsize=(20 , 12))
gs = gridspec.GridSpec(3, 4, figure=fig, wspace=.35, hspace=.35)
ax = fig.add_subplot(gs[0])
plt.scatter(1,1,color = 'grey',s=50,label = '$l$')
plt.xlim(2,4)
plt.ylim(2,4)
set_fig_(ax , iflegend=1)
plt.savefig(output_file("Illustration_legend2.svg"),bbox_inches = 'tight')

# Distribution (length and n k) fig

In [ ]:
coauthordf = pd.read_csv(input_file('NobelCoauthors_2024.csv.zip'))

In [ ]:
laureates = pd.read_csv(BUNDLED_LAUREATES)
df = pd.read_csv(input_file('Author_beginyear.csv.zip'))
laureates['FirstYear'] = laureates['AuthorID'].map(df.set_index('AuthorID')['FirstYear'])
laureates['Prize t'] = laureates['Prize year'] - laureates['FirstYear']

In [ ]:
data = coauthordf.groupby(['Focal_AID','AuthorID'])['t'].min().reset_index()
data.columns = ['Focal_AID','AuthorID','t_start']

maxyear = coauthordf.groupby(['Focal_AID','AuthorID'])['t'].max().reset_index()
data['t_end'] = maxyear['t']
data['l'] = data['t_end'] - data['t_start']+1

n_k = coauthordf.groupby(['Focal_AID','AuthorID'])['PaperID'].nunique().reset_index()
data['k'] = n_k['PaperID']

data['Field'] = data['Focal_AID'].map(laureates.set_index('AuthorID')['Field'])
data = data[data['k']>1].reset_index(drop=True)

In [ ]:
data['Avg_l'] = data.groupby('Focal_AID')['l'].transform('mean')
data['Avg_n_k'] = data.groupby('Focal_AID')['k'].transform('mean')
data['R_l'] = data['l'] / data['Avg_l']
data['R_n_k'] = data['k'] / data['Avg_n_k']

In [ ]:
fig = plt.figure(figsize=(20 , 12))
gs = gridspec.GridSpec(3, 4, figure=fig, wspace=.35, hspace=.35)

data_ = data[data['Field']=='Physics']
ax = fig.add_subplot(gs[0])
sns.histplot(data_['l'].values, binwidth=1, stat='proportion' , color = 'tab:orange')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$l$', ylabel='P ($l$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[1])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_l'].values, binwidth=1, stat='proportion' , color = 'tab:orange')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{l}$', ylabel=r'P ($\hat{l}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[2])
hist_values, bin_edges = np.histogram(data_['R_l'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:orange', marker='D', label='Real data')
log_hist_values = np.log10(hist_values + 1e-6)  # Avoid log of zero by adding a small constant
regression = scipy.stats.linregress(bin_centers, log_hist_values)
fitted_line = 10 ** (bin_centers * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-linear fit')
plt.ylim(.00001 , 1.01)
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$l$ / $\hat{l}$', ylabel=r'P ($l$ / $\hat{l}$)', xlog=0, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[3])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_l' , lw = 2 , color = 'tab:orange', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Physics' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$l$ / $\hat{l}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 1)

data_ = data[data['Field']=='Chemistry']
ax = fig.add_subplot(gs[4])
sns.histplot(data_['l'].values, binwidth=1, stat='proportion' , color = 'tab:blue')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$l$', ylabel='P (l)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[5])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_l'].values, binwidth=1, stat='proportion' , color = 'tab:blue')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{l}$', ylabel=r'P ($\hat{l}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[6])
hist_values, bin_edges = np.histogram(data_['R_l'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:blue', marker='D', label='Real data')
log_hist_values = np.log10(hist_values + 1e-6)  # Avoid log of zero by adding a small constant
regression = scipy.stats.linregress(bin_centers, log_hist_values)
fitted_line = 10 ** (bin_centers * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-linear fit')
plt.ylim(.00001 , 1.01)
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$l$ / $\hat{l}$', ylabel=r'P ($l$ / $\hat{l}$)', xlog=0, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[7])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_l' , lw = 2 , color = 'tab:blue', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Chemistry' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$l$ / $\hat{l}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 1)

data_ = data[data['Field']=='Medicine']
ax = fig.add_subplot(gs[8])
sns.histplot(data_['l'].values, binwidth=1, stat='proportion' , color = 'tab:green')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$l$', ylabel='P (l)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[9])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_l'].values, binwidth=1, stat='proportion' , color = 'tab:green')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{l}$', ylabel=r'P ($\hat{l}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[10])
hist_values, bin_edges = np.histogram(data_['R_l'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:green', marker='D', label='Real data')
log_hist_values = np.log10(hist_values + 1e-6)  # Avoid log of zero by adding a small constant
regression = scipy.stats.linregress(bin_centers, log_hist_values)
fitted_line = 10 ** (bin_centers * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-linear fit')
plt.ylim(.00001 , 1.01)
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$l$ / $\hat{l}$', ylabel=r'P ($l$ / $\hat{l}$)', xlog=0, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[11])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_l' , lw = 2 , color = 'tab:green', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Medicine' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$l$ / $\hat{l}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 1)
plt.savefig(output_file("Distribution_clength.svg"),bbox_inches = 'tight')

In [ ]:
fig = plt.figure(figsize=(20 , 12))
gs = gridspec.GridSpec(3, 4, figure=fig, wspace=.35, hspace=.35)

data_ = data[data['Field']=='Physics']
ax = fig.add_subplot(gs[0])
sns.histplot(data_['k'].values, binwidth=1, stat='proportion' , color = 'tab:orange')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$k$', ylabel='P ($k$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[1])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_n_k'].values, binwidth=1, stat='proportion' , color = 'tab:orange')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{k}$', ylabel=r'P ($\hat{k}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[2])
hist_values, bin_edges = np.histogram(data_['R_n_k'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:orange', marker='D', label='Real data')
regression = scipy.stats.linregress(np.log10(bin_centers+.00001), np.log10(hist_values+.00001))
fitted_line = 10** (np.log10(bin_centers+.00001) * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-log fit')
plt.ylim(.00001 , 1.01)
plt.xlim(.9,10001)
set_fig_(ax , xlabel=r'$k$ / $\hat{k}$', ylabel=r'P ($k$ / $\hat{k}$)', xlog=1, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[3])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_n_k' , lw = 2 , color = 'tab:orange', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Physics' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
plt.ylim(0.3 , 2)
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$k$ / $\hat{k}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 3)

data_ = data[data['Field']=='Chemistry']
ax = fig.add_subplot(gs[4])
sns.histplot(data_['k'].values, binwidth=1, stat='proportion' , color = 'tab:blue')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$k$', ylabel='P ($k$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[5])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_n_k'].values, binwidth=1, stat='proportion' , color = 'tab:blue')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{k}$', ylabel=r'P ($\hat{k}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[6])
hist_values, bin_edges = np.histogram(data_['R_n_k'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:blue', marker='D', label='Real data')
regression = scipy.stats.linregress(np.log10(bin_centers+.00001), np.log10(hist_values+.00001))
fitted_line = 10** (np.log10(bin_centers+.00001) * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-log fit')
plt.ylim(.00001 , 1.01)
plt.xlim(.9,10001)
set_fig_(ax , xlabel=r'$k$ / $\hat{k}$', ylabel=r'P ($k$ / $\hat{k}$)', xlog=1, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[7])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_n_k' , lw = 2 , color = 'tab:blue', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Chemistry' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
plt.ylim(0.3 , 2)
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$k$ / $\hat{k}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 3)

data_ = data[data['Field']=='Medicine']
ax = fig.add_subplot(gs[8])
sns.histplot(data_['k'].values, binwidth=1, stat='proportion' , color = 'tab:green')
plt.xlim(-.5,20)
set_fig_(ax , xlabel='$k$', ylabel='P ($k$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[9])
sns.histplot(data_.drop_duplicates('Focal_AID')['Avg_n_k'].values, binwidth=1, stat='proportion' , color = 'tab:green')
plt.xlim(-.5,15)
set_fig_(ax , xlabel=r'$\hat{k}$', ylabel=r'P ($\hat{k}$)', xlog=0, ylog=0)

ax = fig.add_subplot(gs[10])
hist_values, bin_edges = np.histogram(data_['R_n_k'].values, bins=30, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(bin_centers, hist_values, color='tab:green', marker='D', label='Real data')
regression = scipy.stats.linregress(np.log10(bin_centers+.00001), np.log10(hist_values+.00001))
fitted_line = 10** (np.log10(bin_centers+.00001) * regression.slope + regression.intercept)
plt.plot(bin_centers, fitted_line, color='tab:red', linestyle='-', lw = 2, label='Log-log fit')
plt.ylim(.00001 , 1.01)
plt.xlim(.9,10001)
set_fig_(ax , xlabel=r'$k$ / $\hat{k}$', ylabel=r'P ($k$ / $\hat{k}$)', xlog=1, ylog=1 , iflegend = 1 , legendloc= 1)

ax = fig.add_subplot(gs[11])
sns.lineplot(data_[data_['t_start']<=48] , x = 't_start' , y = 'R_n_k' , lw = 2 , color = 'tab:green', n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.axhline(1 , c = 'grey' , ls='-')
plt.axvline(laureates.loc[laureates['Field']=='Medicine' , 'Prize t'].mean() , c = 'tab:red' , lw = 2 , ls='--' , label = 'Avg Nobel Prize\nwinning age')
set_fig_(ax , xlabel=r'Career age (t)', ylabel=r'$k$ / $\hat{k}$', xlog=0, ylog=0 , iflegend = 1 , legendloc= 3)
plt.ylim(0.3 , 2)
plt.savefig(output_file("Distribution_ck.svg"),bbox_inches = 'tight')

# Collaboration Life Cycle

In [ ]:
data = coauthordf.groupby(['Focal_AID','AuthorID'])['t_start'].min().reset_index()
maxyear = coauthordf.groupby(['Focal_AID','AuthorID'])['t_start'].max().reset_index()
data['t_end'] = maxyear['t_start']
data['l'] = data['t_end'] - data['t_start']+1

n_k = coauthordf.groupby(['Focal_AID','AuthorID'])['PaperID'].nunique().reset_index()
data['k'] = n_k['PaperID']

data['Field'] = data['Focal_AID'].map(laureates.set_index('AuthorID')['Field'])
data = data[data['k']>1].reset_index(drop=True)

In [ ]:
data['Avg_l'] = data.groupby('Focal_AID')['l'].transform('mean')
data['Avg_n_k'] = data.groupby('Focal_AID')['k'].transform('mean')
data['R_l'] = data['l'] / data['Avg_l']
data['R_n_k'] = data['k'] / data['Avg_n_k']

In [ ]:
yn_k = coauthordf.groupby(['Focal_AID', 'AuthorID' , 't_start'])['PaperID'].nunique().reset_index()
yn_k.columns = ['Focal_AID', 'AuthorID' , 't_', 'delta_k']
yn_k = yn_k.merge(data[['Focal_AID', 'AuthorID', 't_start', 'k']] , on = ['Focal_AID', 'AuthorID'])
yn_k['tau'] = yn_k['t_'] - yn_k['t_start']
yn_k['Field'] =yn_k['Focal_AID'].map(laureates.set_index('AuthorID')['Field'])
yn_k = yn_k.sort_values(['Focal_AID', 'AuthorID', 't_']).reset_index(drop=True)
yn_k['cum_k'] = yn_k.groupby(['Focal_AID','AuthorID'])['delta_k'].cumsum()

In [ ]:
def get_t_type(x):
    if x<30:
        return 'Early collaborators'
    else:
        return 'Later collaborators'
yn_k['tbin'] = yn_k['t_start'].apply(get_t_type)

In [ ]:
yn_k_ = yn_k[yn_k['Field'] == 'Physics']

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.lineplot(yn_k_[yn_k_['tau']<=10] , x = 'tau' , y = 'delta_k' , hue = 'tbin', markers=['D'], n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.savefig(output_file('Collaboration_life_cycle.svg'), bbox_inches='tight')

# Relationship with citation/innovation

In [ ]:
metrics = coauthordf.groupby(['Focal_AID', 'AuthorID'], as_index=False)[
    ['Team_Size', 'Novelty_pct', 'cit_pct', 'CD_pct']
].mean()
data = data.merge(metrics, on=['Focal_AID', 'AuthorID'], how='left', validate='one_to_one')

In [ ]:
colors = sns.color_palette("icefire",5)
fig = plt.figure(figsize=(12 , 8))
gs = gridspec.GridSpec(2, 2, figure=fig)
ax = fig.add_subplot(gs[0])
plt.scatter(data['l'].values , data['k'].values , c = 'k' , alpha = .05 , s = data['k'].values)
sns.regplot(data , x='l' , y='k' , scatter=False , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.ylim(-10,200)
set_fig_(ax , xlabel='$l$', ylabel='N k collaboration', xlog=0, ylog=0)

ax = fig.add_subplot(gs[1])
plt.scatter(data['l'].values , data['cit_pct'].values , c = 'k' , alpha = .05 , s = data['k'].values)
sns.regplot(data , x='l' , y='cit_pct' , scatter=False , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
set_fig_(ax , xlabel='$l$', ylabel='Avg citation pct.', xlog=0, ylog=0)

ax = fig.add_subplot(gs[2])
sns.lineplot(data , x='l' , y='k' , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
set_fig_(ax , xlabel='$l$', ylabel='N k collaboration', xlog=0, ylog=0)

ax = fig.add_subplot(gs[3])
sns.lineplot(data , x='l' , y='cit_pct' , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.ylim(-0.05,1.05)
set_fig_(ax , xlabel='$l$', ylabel='Avg citation pct.', xlog=0, ylog=0)
plt.savefig(output_file('Collaboration_citations.svg'), bbox_inches='tight')

In [ ]:
colors = sns.color_palette("icefire",5)
fig = plt.figure(figsize=(12 , 8))
gs = gridspec.GridSpec(2, 2, figure=fig)
ax = fig.add_subplot(gs[0])
sns.lineplot(data , x='l' , y='Novelty_pct' , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
set_fig_(ax , xlabel='$l$', ylabel='Avg novelty pct.', xlog=0, ylog=0)

ax = fig.add_subplot(gs[1])
sns.lineplot(data , x='l' , y='CD_pct' , color = 'tab:red' , n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
set_fig_(ax , xlabel='$l$', ylabel='Avg disruption pct.', xlog=0, ylog=0)
plt.savefig(output_file('Collaboration_innovation.svg'), bbox_inches='tight')